# RNA3D thesis composite-TBM submission

Offline production inference using the same temporal-safe/composite-search code as the local experiments.

In [ ]:
from pathlib import Path
import shutil
import sys
import tarfile
import time

import pandas as pd

ARTIFACT_ARCHIVE = Path('/kaggle/input/rna3d-thesis-inference-artifacts/rna3d_bundle.tar.gz')
BUNDLE = Path('/kaggle/working/rna3d_bundle')
WORK = Path('/kaggle/working/rna3d_work')
COMPETITION = Path('/kaggle/input/stanford-rna-3d-folding')

assert ARTIFACT_ARCHIVE.is_file(), ARTIFACT_ARCHIVE
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)
with tarfile.open(ARTIFACT_ARCHIVE, 'r:gz') as archive:
    archive.extractall(BUNDLE)
sys.path.insert(0, str(BUNDLE))
print('bundle ready:', BUNDLE)

In [ ]:
from kaggle.inference_pipeline import run_inference

test = pd.read_csv(COMPETITION / 'test_sequences.csv')
sample = pd.read_csv(COMPETITION / 'sample_submission.csv')
started = time.time()
submission = run_inference(
    test,
    BUNDLE,
    work_dir=WORK,
    sample_submission=sample,
    steps=300,
    max_len=1000,
)
output = Path('/kaggle/working/submission.csv')
submission.to_csv(output, index=False)
assert submission.shape == sample.shape, (submission.shape, sample.shape)
assert submission['ID'].tolist() == sample['ID'].tolist()
assert not submission.isna().any().any()
print(f'wrote {output}: rows={len(submission)}, targets={len(test)}, seconds={time.time()-started:.1f}')

In [ ]:
submission.head(3)